# Partie D : système SAV multi-agents
## 13a : agent_garantie et outil local consulter_garantie

In [1]:
import calendar
import json
from datetime import date, datetime
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv(override=True)


def trouver_racine() -> Path:
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "pyproject.toml").exists():
            return d
    raise FileNotFoundError("pyproject.toml introuvable")


RACINE = trouver_racine()
REGISTRE = RACINE / "sav" / "registre_garanties.json"


def ajouter_mois(d: date, n: int) -> date:
    mois = d.month - 1 + n
    annee, mois = d.year + mois // 12, mois % 12 + 1
    jour = min(d.day, calendar.monthrange(annee, mois)[1])
    return date(annee, mois, jour)


@tool
def consulter_garantie(numero_serie: str) -> dict:
    """Consulte le registre des garanties à partir d'un numéro de série (ex: PX200-0042).
    Renvoie le produit, la date d'achat, la date de fin et si la garantie est active."""
    with open(REGISTRE, encoding="utf-8") as f:  # lecture seule
        registre = json.load(f)
    for ligne in registre:
        if ligne["numero_serie"].upper() == numero_serie.strip().upper():
            achat = datetime.strptime(ligne["date_achat"], "%Y-%m-%d").date()
            fin = ajouter_mois(achat, ligne["duree_mois"])
            return {
                "numero_serie": ligne["numero_serie"],
                "produit": ligne["produit"],
                "date_achat": str(achat),
                "date_fin_garantie": str(fin),
                "sous_garantie": date.today() <= fin,
                "date_du_jour": str(date.today()),
            }
    return {"erreur": f"Numéro de série {numero_serie} introuvable dans le registre."}


print(RACINE)
print(REGISTRE.exists())

C:\Users\bouho\Desktop\tp5_agents
True


In [2]:
print(consulter_garantie.invoke({"numero_serie": "PX200-0042"}))
print(consulter_garantie.invoke({"numero_serie": "PX200-0007"}))
print(consulter_garantie.invoke({"numero_serie": "XXX"}))

{'numero_serie': 'PX200-0042', 'produit': 'Imprimante PX-200', 'date_achat': '2025-06-15', 'date_fin_garantie': '2027-06-15', 'sous_garantie': True, 'date_du_jour': '2026-11-28'}
{'numero_serie': 'PX200-0007', 'produit': 'Imprimante PX-200', 'date_achat': '2023-03-10', 'date_fin_garantie': '2025-03-10', 'sous_garantie': False, 'date_du_jour': '2026-11-28'}
{'erreur': 'Numéro de série XXX introuvable dans le registre.'}


In [3]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

agent_garantie = create_agent(
    model=llm,
    tools=[consulter_garantie],
    system_prompt=(
        "Tu es l'agent garantie du service après-vente d'un distributeur d'électronique. "
        "Pour toute question de garantie, appelle l'outil consulter_garantie avec le "
        "numéro de série donné par le client. Réponds en français, de façon courte : "
        "indique si le produit est sous garantie et la date de fin. "
        "N'invente jamais de données : si l'outil renvoie une erreur, dis-le au client."
    ),
    name="agent_garantie",
)

resp = agent_garantie.invoke(
    {"messages": [{"role": "user", "content":
        "Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042."}]}
)
for m in resp["messages"]:
    m.pretty_print()

================================ Human Message =================================

Est-ce que mon imprimante est encore sous garantie ? Son numéro de série est PX200-0042.
================================== Ai Message ==================================
Name: agent_garantie
Tool Calls:
  consulter_garantie (7ec865b5-1b17-4aa2-8866-e9075d2763dd)
 Call ID: 7ec865b5-1b17-4aa2-8866-e9075d2763dd
  Args:
    numero_serie: PX200-0042
================================= Tool Message =================================
Name: consulter_garantie

{"numero_serie": "PX200-0042", "produit": "Imprimante PX-200", "date_achat": "2025-06-15", "date_fin_garantie": "2027-06-15", "sous_garantie": true, "date_du_jour": "2026-11-28"}
================================== Ai Message ==================================
Name: agent_garantie

Oui, votre imprimante est encore sous garantie jusqu'au 15 juin 2027.


## 13b : orchestrateur LangGraph (routage vers un agent spécialisé)

In [4]:
from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class EtatSAV(TypedDict):
    messages: Annotated[list, add_messages]
    image_ref: Optional[str]
    categorie: Optional[str]
    reponse_finale: Optional[str]


class Classification(BaseModel):
    categorie: Literal["documentation", "garantie"] = Field(
        description="documentation = panne, voyant, pictogramme, utilisation, manuel ; "
                    "garantie = durée ou statut de garantie d'un produit (numéro de série)."
    )


llm_routeur = ChatOllama(model="qwen3:4b", temperature=0, reasoning=False)
routeur = llm_routeur.with_structured_output(Classification)

PROMPT_ROUTEUR = (
    "Tu es l'orchestrateur du SAV d'un distributeur d'électronique. "
    "Classe la demande du client dans UNE catégorie : "
    "'documentation' (question technique, voyant, pictogramme, dépannage, manuel) "
    "ou 'garantie' (le produit est-il sous garantie, numéro de série)."
)


def orchestrateur(state: EtatSAV):
    question = state["messages"][-1].content
    if state.get("image_ref"):
        question += f"\n(Photo jointe : {state['image_ref']})"
    res = routeur.invoke([("system", PROMPT_ROUTEUR), ("human", question)])
    return {"categorie": res.categorie}


def noeud_documentaire(state: EtatSAV):
    msg = "[agent_documentaire : sera branché à l'étape 13d]"
    return {"messages": [AIMessage(content=msg, name="agent_documentaire")],
            "reponse_finale": msg}


def noeud_garantie(state: EtatSAV):
    res = agent_garantie.invoke({"messages": state["messages"]})
    rep = res["messages"][-1].content
    return {"messages": [AIMessage(content=rep, name="agent_garantie")],
            "reponse_finale": rep}


g = StateGraph(EtatSAV)
g.add_node("orchestrateur", orchestrateur)
g.add_node("agent_documentaire", noeud_documentaire)
g.add_node("agent_garantie", noeud_garantie)

g.add_edge(START, "orchestrateur")
g.add_conditional_edges(
    "orchestrateur",
    lambda s: s["categorie"],
    {"documentation": "agent_documentaire", "garantie": "agent_garantie"},
)
g.add_edge("agent_documentaire", END)
g.add_edge("agent_garantie", END)

graphe_sav = g.compile()
print("Graphe SAV compilé")

Graphe SAV compilé


In [5]:
def demander(texte, image_ref=None):
    res = graphe_sav.invoke(
        {"messages": [HumanMessage(texte)], "image_ref": image_ref}
    )
    print("Catégorie :", res["categorie"])
    print("Réponse   :", res["reponse_finale"])
    print("-" * 60)


demander(
    "Mon imprimante PX-200 affiche un voyant clignotant. "
    "Que signifie le pictogramme sur la photo jointe ?",
    image_ref="img_px200_01",
)
demander(
    "Est-ce que mon imprimante est encore sous garantie ? "
    "Son numéro de série est PX200-0042."
)

Catégorie : documentation
Réponse   : [agent_documentaire : sera branché à l'étape 13d]
------------------------------------------------------------
Catégorie : garantie
Réponse   : Oui, votre imprimante est encore sous garantie jusqu'au 15 juin 2027.
------------------------------------------------------------


## 13c : RAG multimodal (texte + images) sur le manuel PX-200

In [6]:
from tp5_agents.sav_rag import rechercher

for question in [
    "voyant clignotant, pictogramme feuille barrée d'une croix rouge",
    "comment dégager un bourrage papier ?",
    "la cartouche d'encre est vide, que faire ?",
]:
    print("QUESTION :", question)
    for d in rechercher(question, k=3):
        extrait = d.page_content[:110].replace("\n", " ")
        print(f"  [{d.metadata['type']:5}] page {d.metadata['page']} | {extrait}")
    print()

QUESTION : voyant clignotant, pictogramme feuille barrée d'une croix rouge
  [image] page 5 | [Image, page 5, section « 4. Pictogramme : bac à papier vide »] L'image montre un symbole d'avertissement. Il 
  [image] page 4 | [Image, page 4, section « 3. Pictogramme : cartouche d'encre »] L'image montre une goutte d'eau bleue avec un 
  [image] page 3 | [Image, page 3, section « 2. Pictogramme : bourrage papier »] L'image montre un symbole d'interdiction avec un

QUESTION : comment dégager un bourrage papier ?
  [image] page 3 | [Image, page 3, section « 2. Pictogramme : bourrage papier »] L'image montre un symbole d'interdiction avec un
  [texte] page 3 | 2. Pictogramme : bourrage papier Aspect : une feuille barrée d'une croix rouge, avec un coin replié. Significa
  [texte] page 5 | 4. Pictogramme : bac à papier vide Aspect : une pile de feuilles surmontée d'une flèche orange. Signification 

QUESTION : la cartouche d'encre est vide, que faire ?
  [image] page 4 | [Image, page 4, sectio

## 13d : serveur MCP SAV (rag_multimodal) et agent_documentaire

In [7]:
from datetime import timedelta
from langchain_mcp_adapters.client import MultiServerMCPClient

client_sav = MultiServerMCPClient(
    {
        "sav": {
            "transport": "streamable_http",
            "url": "http://127.0.0.1:23001/mcp",
            "timeout": timedelta(minutes=10),
            "sse_read_timeout": timedelta(minutes=10),
        }
    }
)

tools_sav = await client_sav.get_tools()
print([t.name for t in tools_sav])

['rag_multimodal']


In [9]:
# --- agent_documentaire : utilise l'outil distant rag_multimodal (via MCP) ---
agent_documentaire = create_agent(
    model=llm,
    tools=tools_sav,  # [rag_multimodal] récupéré par le client MCP
    system_prompt=(
        "Tu es l'agent documentation du SAV d'un distributeur d'électronique. "
        "Pour toute question technique (voyant, pictogramme, panne, utilisation), "
        "appelle l'outil rag_multimodal avec la question du client. "
        "Si une référence de photo est donnée, passe-la dans le paramètre image_ref. "
        "Réponds en français en reprenant la réponse de l'outil, avec ses sources."
    ),
    name="agent_documentaire",
)


async def noeud_documentaire(state: EtatSAV):
    question = state["messages"][-1].content
    if state.get("image_ref"):
        question += f"\n\nRéférence de la photo jointe : {state['image_ref']}"
    res = await agent_documentaire.ainvoke({"messages": [HumanMessage(question)]})
    rep = res["messages"][-1].content
    return {"messages": [AIMessage(content=rep, name="agent_documentaire")],
            "reponse_finale": rep}


# --- graphe complet (même structure qu'en 13b, avec le vrai agent documentaire) ---
g2 = StateGraph(EtatSAV)
g2.add_node("orchestrateur", orchestrateur)
g2.add_node("agent_documentaire", noeud_documentaire)
g2.add_node("agent_garantie", noeud_garantie)

g2.add_edge(START, "orchestrateur")
g2.add_conditional_edges(
    "orchestrateur",
    lambda s: s["categorie"],
    {"documentation": "agent_documentaire", "garantie": "agent_garantie"},
)
g2.add_edge("agent_documentaire", END)
g2.add_edge("agent_garantie", END)

graphe_sav = g2.compile()
print("Graphe SAV complet compilé")

Graphe SAV complet compilé


In [12]:
async def demander(texte, image_ref=None):
    res = await graphe_sav.ainvoke(
        {"messages": [HumanMessage(texte)], "image_ref": image_ref}
    )
    print("Catégorie :", res["categorie"])
    print("Agent     :", res["messages"][-1].name)
    print("Réponse   :\n", res["reponse_finale"])
    print("=" * 60)


await demander(
    "Mon imprimante PX-200 affiche un voyant clignotant. "
    "Que signifie le pictogramme sur la photo jointe ?",
    image_ref="img_px200_01",
)
await demander(
    "Est-ce que mon imprimante est encore sous garantie ? "
    "Son numéro de série est PX200-0042."
)

Catégorie : documentation
Agent     : agent_documentaire
Réponse   :
 Le pictogramme indique un **« Papier bloqué »** (paper jam). 

Sources : Manuel PX-200, page 3, figure 2
Catégorie : garantie
Agent     : agent_garantie
Réponse   :
 Oui, votre imprimante est encore sous garantie jusqu'au 15 juin 2027.
